# Obtain AlphaEarth Features For An AOI

This notebook only does one job:

1. Read an AOI polygon.
2. Export Google Satellite Embedding / AlphaEarth bands from Earth Engine.
3. Align the exported multiband GeoTIFF to a local template raster grid.

No ML is included here.

## 1. Setup Paths

In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# Change these three paths for a different site.
OUT_DIR = Path("/mnt/c/Users/amehedi/Downloads/thomas")
AOI_PATH = OUT_DIR / "montecito_aoi.shp"
TEMPLATE_PATH = OUT_DIR / "topographic__elevation.tif"

# Export settings.
YEAR = 2023
EXPORT_SCALE = 10  # meters. Use 30 for smaller/faster export.
DRIVE_FOLDER = "earthengine"
SITE_NAME = "thomas"

# Local files after export/download.
RAW_PATH = OUT_DIR / f"alphaearth_{YEAR}_{SITE_NAME}_raw.tif"
ALIGNED_PATH = OUT_DIR / f"alphaearth_{YEAR}_{SITE_NAME}_aligned_to_dem.tif"

ALPHAEARTH_BANDS = [f"A{i:02d}" for i in range(64)]

print("AOI:", AOI_PATH)
print("Template:", TEMPLATE_PATH)
print("Raw AlphaEarth:", RAW_PATH)
print("Aligned AlphaEarth:", ALIGNED_PATH)

## 2. Check AOI And Template

In [ ]:
aoi = gpd.read_file(AOI_PATH)

with rasterio.open(TEMPLATE_PATH) as src:
    template_profile = src.profile.copy()
    template_shape = (src.height, src.width)
    template_crs = src.crs
    template_transform = src.transform
    template_bounds = src.bounds

print("AOI CRS:", aoi.crs)
print("Template CRS:", template_crs)
print("Template shape:", template_shape)
print("Template bounds:", template_bounds)

aoi.to_crs(template_crs).plot(facecolor="none", edgecolor="red")
plt.title("AOI")
plt.show()

## 3. Start Earth Engine Export

This starts an export task to Google Drive. After the task finishes, download the GeoTIFF from Drive and place it at `RAW_PATH`.

In [ ]:
def initialize_earth_engine():
    import ee

    try:
        ee.Initialize()
    except Exception:
        ee.Authenticate()
        ee.Initialize()

    return ee


def aoi_to_ee_geometry(ee):
    aoi_ll = gpd.read_file(AOI_PATH).to_crs(4326)
    geom = aoi_ll.geometry.union_all() if hasattr(aoi_ll.geometry, "union_all") else aoi_ll.geometry.unary_union
    geojson = json.loads(gpd.GeoSeries([geom], crs="EPSG:4326").to_json())["features"][0]["geometry"]
    return ee.Geometry(geojson)


def start_alphaearth_export():
    ee = initialize_earth_engine()
    aoi_ee = aoi_to_ee_geometry(ee)

    with rasterio.open(TEMPLATE_PATH) as template:
        target_crs = template.crs.to_string()

    image = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
        .filterBounds(aoi_ee)
        .mosaic()
        .select(ALPHAEARTH_BANDS)
        .clip(aoi_ee)
        .toFloat()
    )

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"alphaearth_{YEAR}_{SITE_NAME}_raw",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"alphaearth_{YEAR}_{SITE_NAME}_raw",
        region=aoi_ee,
        crs=target_crs,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
    )

    task.start()
    print("Started task ID:", task.id)
    print("Download from Google Drive folder:", DRIVE_FOLDER)
    print("Expected local path after download:", RAW_PATH)
    return task


# Run this line once to start the export.
# task = start_alphaearth_export()

## 4. Check Running Earth Engine Tasks

In [ ]:
def show_earth_engine_tasks():
    ee = initialize_earth_engine()
    tasks = ee.batch.Task.list()

    for task in tasks[:10]:
        status = task.status()
        print(status.get("description"), status.get("state"), status.get("id"))


# Run this to check export status.
# show_earth_engine_tasks()

## 5. Align Downloaded AlphaEarth GeoTIFF To DEM Grid

Run this after the Earth Engine export finishes and the raw GeoTIFF is downloaded to `RAW_PATH`.

In [ ]:
def align_alphaearth_to_template():
    if not RAW_PATH.exists():
        raise FileNotFoundError(f"Missing raw AlphaEarth file: {RAW_PATH}")

    nodata = -9999.0

    with rasterio.open(TEMPLATE_PATH) as template, rasterio.open(RAW_PATH) as src:
        template_arr = template.read(1)
        template_invalid = ~np.isfinite(template_arr)
        if template.nodata is not None:
            template_invalid |= np.isclose(template_arr, template.nodata)

        profile = template.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="float32",
            count=src.count,
            nodata=nodata,
            compress="deflate",
        )

        with rasterio.open(ALIGNED_PATH, "w", **profile) as dst:
            for band in range(1, src.count + 1):
                out = np.full((template.height, template.width), nodata, dtype="float32")

                reproject(
                    source=rasterio.band(src, band),
                    destination=out,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=template.transform,
                    dst_crs=template.crs,
                    dst_nodata=nodata,
                    resampling=Resampling.bilinear,
                )

                out[template_invalid] = nodata
                dst.write(out, band)

    print("Wrote:", ALIGNED_PATH)


# Run after downloading RAW_PATH.
# align_alphaearth_to_template()

## 6. Inspect Aligned Output

In [ ]:
if ALIGNED_PATH.exists():
    with rasterio.open(ALIGNED_PATH) as src:
        print("bands:", src.count)
        print("shape:", src.height, src.width)
        print("crs:", src.crs)
        print("transform:", src.transform)
        band1 = src.read(1).astype("float32")
        if src.nodata is not None:
            band1 = np.where(np.isclose(band1, src.nodata), np.nan, band1)

    plt.figure(figsize=(7, 5))
    plt.imshow(band1, cmap="viridis")
    plt.colorbar(label="AlphaEarth band A00")
    plt.title("Aligned AlphaEarth Feature A00")
    plt.axis("off")
    plt.show()
else:
    print("Aligned file not found yet:", ALIGNED_PATH)